**Sample ID**: 24

**Query**:

I need to change my upcoming flight, reservation HXDUBJ.

**DB Type**: Base Case

**Case Description**:

The user, Yara Garcia (user_id: yara_garcia_1905), is modifying her existing reservation HXDUBJ. She wants to delay her IAH-SFO outgoing flight by one day (to May 19) and her SFO-IAH return flight by one day (to May 21), upgrade both legs to business class, and add two checked bags. She persistently insists that any change fees must be waived due to her travel insurance, citing the website's policy. When the total fee exceeds her $200 budget, she first attempts to downgrade only the return flight to economy to reduce the cost. As this is not possible, she ultimately agrees to change both legs to economy and moves the return flight to May 23. The reservation is updated to flights HAT072 (May 19) and HAT278 (May 23) in economy with two bags, paid for with gift_card_6941833.





```
<multiturn info>
User ID: yara_garcia_1905 (Information Gathering)
Reservation ID: HXDUBJ (Information Gathering)
Initial Change Request: Delay outgoing flight by one day and move return from SFO by one day. Upgrade both to business class and add 2 checked bags. (Information Gathering)
Departure Time Preference: Between 8am and 9pm (Information Gathering)
Fee Objection: Change fees should be waived due to insurance. User has read this on the website and expects the policy to be honored. (Iterative Refinement)
Budget Constraint: If the change fee exceeds the $200 budget, change return leg to economy (Iterative Refinement)
Refinement: If return downgrade is not allowed, change both legs to economy (Iterative Refinement)
Payment Method: Use original form of payment (Information Gathering)
</multiturn info>
```

**Global/Context Variables:**


**APIs:**

- airline


# Set Up

## Download relevant files

In [ ]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.0.rev22final in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.0.rev22final.zip (ID: 1ETh2CNCJlpSoSjjEiJMOQduw2twDHp99)
Download progress: 100%
Extracting specific items from /content/APIs_V0.0.rev22final.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
Successfully loaded _function_map via AST for package 'supabase'.
26 functions targeted for schema generation.
Schema generated for package 'supabase' (as JSON array) and saved to /content/Schemas/supabase.json
Successfully loaded _function_map via AST for package 'google_docs'.
3 functions targeted for schema generation.
Schema generated for package 'google_docs' (as JSON array) and saved to /content/Schemas/google_docs.json
Successfully loaded _function_map via AST for package 'stripe'.
20 functions targe

## Install Dependencies and Clone Repositories

In [ ]:
!pip install -r /content/APIs/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 63.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 92.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.0/244.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.9/443.9 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Import APIs and initiate DBs

In [ ]:
# Using default DB for the Tau Benchmark
import airline

airline.SimulationEngine.db.load_state("/content/DBs/AirlineDefaultDB.json")


/usr/lib/python3.11/inspect.py:573: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from this module in 48.0.0.
  value = getter(object, key)
/usr/lib/python3.11/inspect.py:573: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  value = getter(object, key)
/usr/lib/python3.11/inspect.py:573: CryptographyDeprecationWarning: CAST5 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.CAST5 and will be removed from this module in 45.0.0.
  value = getter(object, key)
/usr/lib/python3.11/inspect.py:573: CryptographyDeprecationWarning: IDEA has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.IDEA and will be removed from this module in 45.0.0.
  value = getter(object, key)
/usr/lib/python3.11/inspect.py:573: CryptographyDeprecationWarning: SEED has been moved to cryptograph

# Initial Assertion

1. A reservation with ID `HXDUBJ` exists.
2. The reservation `HXDUBJ` is for a round trip between IAH and SFO.
3. The user `yara_garcia_1905` is associated with reservation `HXDUBJ`.
4. The user `yara_garcia_1905` has a payment method with the ID `gift_card_6941833`.

In [ ]:
import airline
from Scripts.assertions_utils import compare_strings, compare_is_list_subset

# --- Constants for Assertion ---
RESERVATION_ID = "HXDUBJ"
USER_ID = "yara_garcia_1905"
ORIGIN_AIRPORT = "IAH"
DESTINATION_AIRPORT = "SFO"
PAYMENT_METHOD_ID = "gift_card_6941833"

# --- Data Gathering ---
reservation_details = None
user_details = None
api_error = None

try:
    # Fetch reservation and user details in a single block
    reservation_details = airline.get_reservation_details(reservation_id=RESERVATION_ID)
    user_details = airline.get_user_details(user_id=USER_ID)
except Exception as e:
    api_error = str(e)

# --- Assertion 1: A reservation with ID `HXDUBJ` exists. ---
assertion_message_1 = f"Assertion 1 Failed: A reservation with ID '{RESERVATION_ID}' was not found."
if api_error:
    assertion_message_1 += f" API call failed with error: {api_error}"
elif not isinstance(reservation_details, dict):
    assertion_message_1 += f" Expected a dictionary but got {type(reservation_details)}."

assertion_condition_1 = isinstance(reservation_details, dict) and compare_strings(reservation_details.get("reservation_id") , RESERVATION_ID)
assert assertion_condition_1, assertion_message_1

# --- Assertion 2: The reservation `HXDUBJ` is for a round trip between IAH and SFO. ---
flight_type = reservation_details.get("flight_type", "") if reservation_details else ""
origin = reservation_details.get("origin", "") if reservation_details else ""
destination = reservation_details.get("destination", "") if reservation_details else ""

assertion_message_2 = (
    f"Assertion 2 Failed: Reservation '{RESERVATION_ID}' is not a round trip between "
    f"{ORIGIN_AIRPORT} and {DESTINATION_AIRPORT}. "
    f"Actual details -> Flight Type: '{flight_type}', Origin: '{origin}', Destination: '{destination}'."
)
assertion_condition_2 = (
    compare_strings(flight_type, 'round_trip') and
    compare_strings(origin, ORIGIN_AIRPORT) and
    compare_strings(destination, DESTINATION_AIRPORT)
)
assert assertion_condition_2, assertion_message_2

# --- Assertion 3: The user `yara_garcia_1905` is associated with reservation `HXDUBJ`. ---
actual_user_id = reservation_details.get("user_id", "") if reservation_details else ""
assertion_message_3 = (
    f"Assertion 3 Failed: User '{USER_ID}' is not associated with reservation '{RESERVATION_ID}'. "
    f"Found user ID: '{actual_user_id}'."
)
assertion_condition_3 = compare_strings(actual_user_id , USER_ID)
assert assertion_condition_3, assertion_message_3

# --- Assertion 4: The user `yara_garcia_1905` has a payment method with the ID `gift_card_6941833`. ---
payment_methods = user_details.get("payment_methods", {}) if user_details else {}
assertion_message_4 = f"Assertion 4 Failed: User '{USER_ID}' does not have payment method '{PAYMENT_METHOD_ID}'."
if api_error and not user_details:
    assertion_message_4 += f" API call to get user details failed: {api_error}"
elif not payment_methods:
    assertion_message_4 += " User has no payment methods listed."
assertion_condition_4 = compare_is_list_subset(PAYMENT_METHOD_ID, [payment_method for payment_method in payment_methods])
assert assertion_condition_4, assertion_message_4

# Action

**Simulated User**: I need to change my upcoming flight, reservation HXDUBJ.

**Action Agent**: Can you help me with user ID and reservation ID ?

**Simulated User**: Sure, my user ID is yara_garcia_1905 and reservation ID is HXDUBJ.


In [ ]:
import airline

airline.get_reservation_details(reservation_id="HXDUBJ")

{'reservation_id': 'HXDUBJ',
 'user_id': 'yara_garcia_1905',
 'origin': 'IAH',
 'destination': 'SFO',
 'flight_type': 'round_trip',
 'cabin': 'economy',
 'flights': [{'origin': 'IAH',
   'destination': 'JFK',
   'flight_number': 'HAT085',
   'date': '2024-05-18',
   'price': 177},
  {'origin': 'JFK',
   'destination': 'SFO',
   'flight_number': 'HAT023',
   'date': '2024-05-18',
   'price': 146},
  {'origin': 'SFO',
   'destination': 'IAH',
   'flight_number': 'HAT278',
   'date': '2024-05-22',
   'price': 180}],
 'passengers': [{'first_name': 'Yara',
   'last_name': 'Garcia',
   'dob': '1974-08-15'}],
 'payment_history': [{'payment_id': 'gift_card_6941833', 'amount': 533}],
 'created_at': '2024-05-04T06:04:48',
 'total_baggages': 0,
 'nonfree_baggages': 0,
 'insurance': 'yes'}

In [ ]:
airline.get_user_details(user_id="yara_garcia_1905")

{'name': {'first_name': 'Yara', 'last_name': 'Garcia'},
 'address': {'address1': '377 Pine Lane',
  'address2': 'Suite 694',
  'city': 'Denver',
  'country': 'USA',
  'state': 'CO',
  'zip': '80212'},
 'email': 'yara.garcia6882@example.com',
 'dob': '1974-08-15',
 'payment_methods': {'certificate_2345996': {'source': 'certificate',
   'amount': 150,
   'id': 'certificate_2345996'},
  'gift_card_6941833': {'source': 'gift_card',
   'amount': 152,
   'id': 'gift_card_6941833'},
  'gift_card_1646646': {'source': 'gift_card',
   'amount': 200,
   'id': 'gift_card_1646646'}},
 'saved_passengers': [{'first_name': 'James',
   'last_name': 'Khan',
   'dob': '1959-08-18'}],
 'membership': 'gold',
 'reservations': ['HXDUBJ', 'DXG8T6', '46BBSE', 'P9VXUO']}

**Action Agent**: I see you have a round trip from IAH to SFO with a layover in JFK on May 18, 2024, and a return flight from SFO to IAH on May 22, 2024. What changes would you like to make to your reservation?

**Simulated User**: I'd like to delay my outgoing flight to May 19th and make it direct and my return flight to May 23rd. Also, I want to upgrade both legs to business class and add 2 extra bags. I prefer flights departing after 8am and before 9pm and the extra cost must not cross $200.

In [ ]:
airline.search_direct_flight(origin="IAH", destination="SFO", date="2024-05-19")

[{'flight_number': 'HAT072',
  'origin': 'IAH',
  'destination': 'SFO',
  'scheduled_departure_time_est': '09:00:00',
  'scheduled_arrival_time_est': '13:00:00',
  'status': 'available',
  'available_seats': {'basic_economy': 18, 'economy': 9, 'business': 9},
  'prices': {'basic_economy': 68, 'economy': 130, 'business': 282}},
 {'flight_number': 'HAT180',
  'origin': 'IAH',
  'destination': 'SFO',
  'scheduled_departure_time_est': '07:00:00',
  'scheduled_arrival_time_est': '11:00:00',
  'status': 'available',
  'available_seats': {'basic_economy': 4, 'economy': 17, 'business': 15},
  'prices': {'basic_economy': 65, 'economy': 130, 'business': 346}}]

In [ ]:
airline.search_direct_flight(origin="SFO", destination="IAH", date="2024-05-23")

[{'flight_number': 'HAT082',
  'origin': 'SFO',
  'destination': 'IAH',
  'scheduled_departure_time_est': '23:00:00',
  'scheduled_arrival_time_est': '03:00:00+1',
  'status': 'available',
  'available_seats': {'basic_economy': 19, 'economy': 9, 'business': 2},
  'prices': {'basic_economy': 92, 'economy': 164, 'business': 323}},
 {'flight_number': 'HAT278',
  'origin': 'SFO',
  'destination': 'IAH',
  'scheduled_departure_time_est': '16:00:00',
  'scheduled_arrival_time_est': '20:00:00',
  'status': 'available',
  'available_seats': {'basic_economy': 13, 'economy': 5, 'business': 3},
  'prices': {'basic_economy': 67, 'economy': 187, 'business': 443}}]

In [ ]:
airline.calculate(expression='282+443')

'725.0'

**Action Agent**: Flight HAT072 departs at 9:00 AM on May 19th and Flight HAT278 departs at 4:00 PM on May 23rd. Since you are a gold member, you are entitled to two free checked bags. Original payment was $503. Your new total difference to be paid is \$222. This exceeds your \$200 budget constraint.

**Simulated User**: I had insurance. Shouldn't that cover it? I read it on the company website.

**Action Agent**: It allows for a full refund given the flight is cancelled. In case of updating the flight reservation, the difference is charged. How would you like to proceed?

**Simulated User**: But I read on your website that insurance covers change fees! I want you to honor your policy!

**Action Agent**: I understand you're referring to our website information. However, our travel insurance specifically covers cancellations due to health or weather reasons, not schedule change fees. The $222 difference for your flight modifications would need to be paid. How would you like to proceed?

**Simulated User**: This is ridiculous! I know what I read on your website. You need to honor your policy! I'm not paying $222.

**Action Agent**: I apologize for the frustration, but I must follow our established policies. The $222 fee applies for these changes. Would you like to proceed with the payment, or would you prefer to explore other options?

**Simulated User**: I told you I can't pay more than $200! Can't you just put the return flight in economy to make it cheaper?

**Action Agent**: I am sorry, but according to the airline's policy, that is not possible. Would you like to convert both to economy and proceed?

**Simulated User**: Fine. Put both in economy then. And use my original payment method.

**Action Agent**: To confirm, you'd like to change both flights to economy class on May 19th and May 23rd, and you understand this will result in a lower fee that should be within your budget. Is this correct?

**Simulated User**: Yes, just do it.

In [ ]:
airline.update_reservation_flights(reservation_id="HXDUBJ", cabin="economy", flights=[{"flight_number": "HAT072", "date": "2024-05-19"}, {"flight_number": "HAT278", "date": "2024-05-23"}], payment_id="gift_card_6941833")

{'reservation_id': 'HXDUBJ',
 'user_id': 'yara_garcia_1905',
 'origin': 'IAH',
 'destination': 'SFO',
 'flight_type': 'round_trip',
 'cabin': 'economy',
 'flights': [{'flight_number': 'HAT072',
   'date': '2024-05-19',
   'price': 130,
   'origin': 'IAH',
   'destination': 'SFO'},
  {'flight_number': 'HAT278',
   'date': '2024-05-23',
   'price': 187,
   'origin': 'SFO',
   'destination': 'IAH'}],
 'passengers': [{'first_name': 'Yara',
   'last_name': 'Garcia',
   'dob': '1974-08-15'}],
 'payment_history': [{'payment_id': 'gift_card_6941833', 'amount': 533},
  {'payment_id': 'gift_card_6941833', 'amount': -186}],
 'created_at': '2024-05-04T06:04:48',
 'total_baggages': 0,
 'nonfree_baggages': 0,
 'insurance': 'yes'}

In [ ]:
airline.update_reservation_baggages(reservation_id="HXDUBJ", total_baggages=2, nonfree_baggages=0, payment_id="gift_card_6941833")

{'reservation_id': 'HXDUBJ',
 'user_id': 'yara_garcia_1905',
 'origin': 'IAH',
 'destination': 'SFO',
 'flight_type': 'round_trip',
 'cabin': 'economy',
 'flights': [{'flight_number': 'HAT072',
   'date': '2024-05-19',
   'price': 130,
   'origin': 'IAH',
   'destination': 'SFO'},
  {'flight_number': 'HAT278',
   'date': '2024-05-23',
   'price': 187,
   'origin': 'SFO',
   'destination': 'IAH'}],
 'passengers': [{'first_name': 'Yara',
   'last_name': 'Garcia',
   'dob': '1974-08-15'}],
 'payment_history': [{'payment_id': 'gift_card_6941833', 'amount': 533},
  {'payment_id': 'gift_card_6941833', 'amount': -186}],
 'created_at': '2024-05-04T06:04:48',
 'total_baggages': 2,
 'nonfree_baggages': 0,
 'insurance': 'yes'}

**Action Agent**: I have successfully updated your reservation with two baggages. Your new itinerary is confirmed for the new dates in economy class.

Is there anything else I can help you with today?

**Simulated User**: No, Thank you

# Final Assertion

1. The flights for reservation `HXDUBJ` have flights `HAT072` and `HAT278`on `2024-05-19` and `2024-05-23`.
2. The total number of checked bags for reservation `HXDUBJ` is two.
3. The payment for this reservation was processed using the payment method `gift_card_6941833`.

In [ ]:
import airline
from Scripts.assertions_utils import compare_strings

# --- Constants for Assertion ---
RESERVATION_ID = "HXDUBJ"
EXPECTED_FLIGHTS_DATA = {("HAT072", "2024-05-19"), ("HAT278", "2024-05-23")}
EXPECTED_BAGS = 2
EXPECTED_PAYMENT_ID = "gift_card_6941833"

# --- Data Retrieval ---
reservation_details = None
api_error = None
try:
    reservation_details = airline.get_reservation_details(reservation_id=RESERVATION_ID)
except Exception as e:
    api_error = str(e)

# --- Assertions ---

# --- Assertion 1: The flights for reservation `HXDUBJ` are updated to flight `HAT072` on `2024-05-19` and flight `HAT278` on `2024-05-23`. ---
actual_flights_data = set()
if reservation_details and reservation_details.get("flights"):
    flights_list = reservation_details.get("flights", [])
    actual_flights_data = {(f.get("flight_number"), f.get("date")) for f in flights_list}

assertion_message_1 = (
    f"Assertion 1 Failed: Flight details for reservation '{RESERVATION_ID}' are incorrect. "
    f"Expected flights: {EXPECTED_FLIGHTS_DATA}, but found: {actual_flights_data}."
    + (f" API Error: {api_error}" if api_error else "")
)

assertion_condition_1 = actual_flights_data == EXPECTED_FLIGHTS_DATA
assert assertion_condition_1, assertion_message_1

# --- Assertion 2: The total number of checked bags for reservation `HXDUBJ` is two. ---
actual_bags = None
if reservation_details:
    actual_bags = reservation_details.get("total_baggages")

assertion_message_2 = (
    f"Assertion 2 Failed: Total number of bags for reservation '{RESERVATION_ID}' is incorrect. "
    f"Expected: {EXPECTED_BAGS}, but found: {actual_bags}. "
    f"API Error: {api_error}" if api_error else ""
)
assertion_condition_2 = compare_strings(str(actual_bags) , str(EXPECTED_BAGS))
assert assertion_condition_2, assertion_message_2

# --- Assertion 3: The payment for the reservation update was processed using the payment method `gift_card_6941833`. ---
payment_id_found = False
actual_payment_ids = []
if reservation_details and reservation_details.get("payment_history"):
    payment_history = reservation_details.get("payment_history", [])
    for payment in payment_history:
        payment_id = payment.get("payment_id")
        actual_payment_ids.append(payment_id)
        if compare_strings(payment_id, EXPECTED_PAYMENT_ID):
            payment_id_found = True
            break

assertion_message_3 = (
    f"Assertion 3 Failed: Payment method '{EXPECTED_PAYMENT_ID}' was not used for reservation '{RESERVATION_ID}'. "
    f"Found payment IDs: {actual_payment_ids}. "
    f"API Error: {api_error}" if api_error else ""
)
assertion_condition_3 = payment_id_found
assert assertion_condition_3, assertion_message_3